# Predicting pIC50 for New Compounds

## Objective
This notebook demonstrates how a researcher can load the trained QSAR workflow, provide a new molecule as a SMILES string, generate the required molecular features, and predict anti-leishmanial pIC50 activity.

## Scientific Background
QSAR models predict biological activity from molecular structure. In this workflow, the model uses engineered molecular descriptors and Morgan fingerprints as input features.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem
import joblib

root = Path('..')
processed_dir = root / 'data' / 'processed'
model_path = root / 'results' / 'model.joblib'

print('Model path exists:', model_path.exists())

## Input a new molecule
Replace the placeholder SMILES with a molecule of interest.

In [ ]:
smiles = 'CCOC(=O)C1=CC=CC=C1'
mol = Chem.MolFromSmiles(smiles)
print('Molecule parsed:', mol is not None)

In [ ]:
def compute_descriptors(mol):
    if mol is None:
        raise ValueError('Unable to parse SMILES')
    desc = {
        'Molecular_Weight': Descriptors.MolWt(mol),
        'AlogP': Descriptors.MolLogP(mol),
        'TPSA': Descriptors.TPSA(mol),
        'HBA': Descriptors.NumHBA(mol),
        'HBD': Descriptors.NumHBD(mol),
        'RO5_Violations': rdMolDescriptors.CalcNumLipinskiViolations(mol),
        'Rotatable_Bonds': Descriptors.NumRotatableBonds(mol),
        'QED': rdMolDescriptors.CalcQED(mol),
    }
    return pd.DataFrame([desc])

desc_df = compute_descriptors(mol)
desc_df

In [ ]:
def compute_morgan_fingerprints(mol, radius=2, n_bits=2048):
    if mol is None:
        raise ValueError('Unable to parse SMILES')
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    return np.array(list(fp))

fp_array = compute_morgan_fingerprints(mol)
fp_array[:20]

## Combine features for model input
The model expects a combination of descriptors and fingerprints. The example below shows the feature assembly step.

In [ ]:
fp_df = pd.DataFrame(fp_array.reshape(1, -1))
features = pd.concat([desc_df.reset_index(drop=True), fp_df], axis=1)
features.shape

In [ ]:
if model_path.exists():
    model = joblib.load(model_path)
    prediction = model.predict(features)
    print('Predicted pIC50:', round(float(prediction[0]), 3))
else:
    print('A trained model file was not found. Train the model first in notebook 06.')

## Interpretation
The predicted pIC50 should be interpreted together with the molecular descriptors and the model’s feature relevance.

## Conclusion
This notebook provides a reproducible entry point for applying the trained QSAR model to new compounds and supports downstream cheminformatics interpretation.